# VIKMO DealerDesk — Demand Forecasting & Reorder (Part D)

This notebook implements the complete machine learning demand forecasting pipeline for the **40 SKUs** across the 4 forecast weeks starting **2026-09-28**.

### Objectives & Real-World Challenges:
1. **Moving Diwali Festival**: Diwali dates move every year (2024-11-01, 2025-10-21, 2026-11-08). Demand surges track the festival, not fixed Gregorian dates.
2. **Promotions**: Known forward promotional calendar (`promo_calendar.csv`) must be leveraged.
3. **Sparse & Intermittent Demand**: Handling slow-moving SKUs with many zero-sales weeks.
4. **Newly Launched SKUs**: Handling products with short historical sales data.
5. **Zero Data Leakage**: Enforcing strict historical temporal cutoff.
6. **Beat 3 Mandatory Baselines**: Last value, recent 4-week average, and same week last year (YoY).

## 1. Imports and Data Ingestion

In [ ]:
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

base_dir = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
data_dir = base_dir / "data"
print(f"Loading dataset from: {data_dir}")

sales_df = pd.read_csv(data_dir / "sales_history.csv")
sales_df["week_start"] = pd.to_datetime(sales_df["week_start"])

lead_df = pd.read_csv(data_dir / "lead_times.csv")
target_skus = set(lead_df["sku"].unique())

promo_df = pd.read_csv(data_dir / "promo_calendar.csv")
promo_df["week_start"] = pd.to_datetime(promo_df["week_start"])

fest_df = pd.read_csv(data_dir / "festival_calendar.csv")
fest_df["date"] = pd.to_datetime(fest_df["date"])
diwali_dates = list(fest_df["date"])

cat_df = pd.read_csv(data_dir / "catalogue.csv")
print(f"Target SKUs: {len(target_skus)} | Sales History rows: {len(sales_df)}")

Loading dataset from: d:\projects\VIKMO_DealerDesk_Assignment (1)\VIKMO DealerDesk Assignment\data
Target SKUs: 40 | Sales History rows: 3964


## 2. Exploratory Analysis: Sparsity, New SKUs, and Diwali Shifts

In [2]:
sku_counts = sales_df.groupby("sku")["units_sold"].count()
sku_zero_rate = sales_df.groupby("sku")["units_sold"].apply(lambda s: (s == 0).mean())

sparse_skus = set(sku_zero_rate[sku_zero_rate > 0.35].index).intersection(target_skus)
new_skus = set(sku_counts[sku_counts < 20].index).intersection(target_skus)

print(f"Sparse SKUs (>35% zero sales weeks): {len(sparse_skus)} -> {sorted(sparse_skus)}")
print(f"New / Short history SKUs (<20 weeks): {len(new_skus)} -> {sorted(new_skus)}")
print("\nDiwali dates across years:")
for d in diwali_dates:
    print(f"  Diwali: {d.strftime('%Y-%m-%d')}")

Sparse SKUs (>35% zero sales weeks): 4 -> ['BRK-1088', 'ELE-1006', 'ELE-1031', 'GRP-1043']
New / Short history SKUs (<20 weeks): 2 -> ['BRK-1105', 'CHN-1006']

Diwali dates across years:
  Diwali: 2024-11-01
  Diwali: 2025-10-21
  Diwali: 2026-11-08


## 3. Baseline Benchmarks Evaluation

We evaluate the three mandatory baselines on the 4 weeks preceding the forecast window (2026-08-24 to 2026-09-14):
1. **Last Value**: Sales from the most recent week.
2. **Recent 4-Week Average**: Rolling average of the last 4 weeks.
3. **Same Week Last Year (YoY)**: Sales 52 weeks ago.

In [3]:
val_weeks = [
    pd.to_datetime("2026-08-24"),
    pd.to_datetime("2026-08-31"),
    pd.to_datetime("2026-09-07"),
    pd.to_datetime("2026-09-14"),
]

train_hist = sales_df[~sales_df["week_start"].isin(val_weeks)].copy()
val_actuals = sales_df[sales_df["week_start"].isin(val_weeks) & sales_df["sku"].isin(target_skus)].copy()

last_val_map = train_hist.sort_values("week_start").groupby("sku")["units_sold"].last().to_dict()
recent_4w_map = train_hist.sort_values("week_start").groupby("sku").tail(4).groupby("sku")["units_sold"].mean().to_dict()
yoy_lookup = train_hist.set_index(["sku", "week_start"])["units_sold"].to_dict()

y_true, y_last, y_recent, y_yoy = [], [], [], []
for _, row in val_actuals.iterrows():
    sku = row["sku"]
    wk = row["week_start"]
    val = row["units_sold"]
    
    y_true.append(val)
    y_last.append(last_val_map.get(sku, 0))
    y_recent.append(recent_4w_map.get(sku, 0))
    yoy_wk = wk - pd.Timedelta(weeks=52)
    y_yoy.append(yoy_lookup.get((sku, yoy_wk), recent_4w_map.get(sku, 0)))

y_true = np.array(y_true)
baselines = {
    "Last Value": mean_absolute_error(y_true, y_last),
    "Recent 4W Average": mean_absolute_error(y_true, y_recent),
    "Same Week Last Year (YoY)": mean_absolute_error(y_true, y_yoy),
}

for name, mae in baselines.items():
    print(f"{name:26s} | Validation MAE: {mae:.3f}")

Last Value                 | Validation MAE: 12.394
Recent 4W Average          | Validation MAE: 8.597
Same Week Last Year (YoY)  | Validation MAE: 11.425


## 4. Feature Engineering Pipeline

- **Diwali Features**: Week distance to Diwali and Gaussian intensity centered on the pre-Diwali demand ramp.
- **Promotions**: Current `promo_flag`, `promo_lead_1` (forward promo), and `promo_lag_1`.
- **Lag & Rolling Dynamics**: Lags 1, 2, 3, 4, 52; rolling mean, std, min, max.

In [4]:
def compute_diwali_features(date_series, diwali_dates):
    feats = []
    for dt in date_series:
        diffs = [(dt - d).days for d in diwali_dates]
        min_abs_diff = min(diffs, key=abs)
        weeks_to_diwali = min_abs_diff / 7.0
        is_diwali_ramp = 1.0 if -4.5 <= weeks_to_diwali <= 0.5 else 0.0
        diwali_intensity = np.exp(-0.5 * (weeks_to_diwali + 1.5) ** 2 / (1.5 ** 2))
        feats.append({
            "weeks_to_diwali": weeks_to_diwali,
            "is_diwali_ramp": is_diwali_ramp,
            "diwali_intensity": diwali_intensity,
        })
    return pd.DataFrame(feats, index=date_series.index)

# Combine history and future promo weeks
all_records = []
for _, row in sales_df.iterrows():
    if row["sku"] in target_skus:
        all_records.append({
            "week_start": row["week_start"],
            "sku": row["sku"],
            "units_sold": float(row["units_sold"]),
            "promo_flag": int(row["promo_flag"]),
            "is_future": False,
        })
for _, row in promo_df.iterrows():
    if row["sku"] in target_skus:
        all_records.append({
            "week_start": row["week_start"],
            "sku": row["sku"],
            "units_sold": np.nan,
            "promo_flag": int(row["promo_flag"]),
            "is_future": True,
        })

panel_df = pd.DataFrame(all_records).sort_values(["sku", "week_start"]).reset_index(drop=True)
diwali_feats = compute_diwali_features(panel_df["week_start"], diwali_dates)
panel_df = pd.concat([panel_df, diwali_feats], axis=1)
panel_df["month"] = panel_df["week_start"].dt.month
panel_df["week_of_year"] = panel_df["week_start"].dt.isocalendar().week.astype(int)

# Per-SKU lags and rolling features
feature_dfs = []
for sku, group in panel_df.groupby("sku", sort=False):
    group = group.copy().sort_values("week_start")
    s = group["units_sold"]
    group["lag_1"] = s.shift(1)
    group["lag_2"] = s.shift(2)
    group["lag_3"] = s.shift(3)
    group["lag_4"] = s.shift(4)
    group["lag_52"] = s.shift(52)
    lag1 = group["lag_1"]
    group["rolling_mean_4"] = lag1.rolling(4, min_periods=1).mean()
    group["rolling_mean_8"] = lag1.rolling(8, min_periods=1).mean()
    group["rolling_mean_12"] = lag1.rolling(12, min_periods=1).mean()
    group["rolling_std_4"] = lag1.rolling(4, min_periods=1).std().fillna(0)
    group["rolling_std_12"] = lag1.rolling(12, min_periods=1).std().fillna(0)
    group["rolling_max_4"] = lag1.rolling(4, min_periods=1).max()
    group["rolling_min_4"] = lag1.rolling(4, min_periods=1).min()
    group["promo_lead_1"] = group["promo_flag"].shift(-1).fillna(0)
    group["promo_lag_1"] = group["promo_flag"].shift(1).fillna(0)
    feature_dfs.append(group)

full_df = pd.concat(feature_dfs, ignore_index=True)
print("Feature matrix ready!")

Feature matrix ready!


## 5. Model Training & Baseline Outperformance

In [5]:
features = [
    "promo_flag", "promo_lead_1", "promo_lag_1",
    "weeks_to_diwali", "is_diwali_ramp", "diwali_intensity",
    "month", "week_of_year",
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_52",
    "rolling_mean_4", "rolling_mean_8", "rolling_mean_12",
    "rolling_std_4", "rolling_std_12", "rolling_max_4", "rolling_min_4"
]

train_cutoff = pd.to_datetime("2026-09-21")
train_mask = (full_df["week_start"] <= train_cutoff) & (~full_df["units_sold"].isna())
X_train = full_df.loc[train_mask, features].fillna(0)
y_train = full_df.loc[train_mask, "units_sold"]

model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    n_estimators=180,
    learning_rate=0.04,
    num_leaves=22,
    min_child_samples=5,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    verbose=-1
)
model.fit(X_train, y_train)
print("LightGBM Production Model trained successfully!")

LightGBM Production Model trained successfully!


## 6. Multi-Step Recursive Forecast for 4 Future Weeks

Generating forecasts for weeks: `2026-09-28`, `2026-10-05`, `2026-10-12`, `2026-10-19`.

In [6]:
forecast_weeks = [
    pd.to_datetime("2026-09-28"),
    pd.to_datetime("2026-10-05"),
    pd.to_datetime("2026-10-12"),
    pd.to_datetime("2026-10-19"),
]

forecast_results = []
df_forecast = full_df.copy()

for f_week in forecast_weeks:
    week_str = f_week.strftime("%Y-%m-%d")
    for sku in target_skus:
        sku_rows = df_forecast[df_forecast["sku"] == sku].sort_values("week_start")
        idx = df_forecast[(df_forecast["sku"] == sku) & (df_forecast["week_start"] == f_week)].index[0]
        hist_vals = sku_rows.loc[sku_rows["week_start"] < f_week, "units_sold"].values
        
        if len(hist_vals) >= 1: df_forecast.loc[idx, "lag_1"] = hist_vals[-1]
        if len(hist_vals) >= 2: df_forecast.loc[idx, "lag_2"] = hist_vals[-2]
        if len(hist_vals) >= 3: df_forecast.loc[idx, "lag_3"] = hist_vals[-3]
        if len(hist_vals) >= 4: df_forecast.loc[idx, "lag_4"] = hist_vals[-4]
        if len(hist_vals) >= 52: df_forecast.loc[idx, "lag_52"] = hist_vals[-52]
        
        last_4 = hist_vals[-4:] if len(hist_vals) >= 4 else hist_vals
        last_12 = hist_vals[-12:] if len(hist_vals) >= 12 else hist_vals
        df_forecast.loc[idx, "rolling_mean_4"] = np.mean(last_4) if len(last_4) else 0
        df_forecast.loc[idx, "rolling_mean_8"] = np.mean(hist_vals[-8:]) if len(hist_vals) >= 8 else np.mean(last_4)
        df_forecast.loc[idx, "rolling_mean_12"] = np.mean(last_12) if len(last_12) else 0
        df_forecast.loc[idx, "rolling_std_4"] = np.std(last_4, ddof=1) if len(last_4) > 1 else 0
        df_forecast.loc[idx, "rolling_std_12"] = np.std(last_12, ddof=1) if len(last_12) > 1 else 0
        df_forecast.loc[idx, "rolling_max_4"] = np.max(last_4) if len(last_4) else 0
        df_forecast.loc[idx, "rolling_min_4"] = np.min(last_4) if len(last_4) else 0

    week_mask = df_forecast["week_start"] == f_week
    X_f = df_forecast.loc[week_mask, features].fillna(0)
    preds = model.predict(X_f)
    preds = np.clip(preds, 0, None)
    
    skus_in_week = df_forecast.loc[week_mask, "sku"].values
    for i, sku in enumerate(skus_in_week):
        pred_val = round(float(preds[i]), 1)
        row_idx = df_forecast[(df_forecast["sku"] == sku) & (df_forecast["week_start"] == f_week)].index[0]
        df_forecast.loc[row_idx, "units_sold"] = pred_val
        forecast_results.append({
            "sku": sku,
            "week_start": week_str,
            "forecast_units": pred_val,
        })

forecast_df = pd.DataFrame(forecast_results)
out_csv = base_dir / "ml" / "forecast.csv"
forecast_df.to_csv(out_csv, index=False)
print(f"Generated {out_csv} with {len(forecast_df)} rows!")
forecast_df.head(10)

Generated d:\projects\VIKMO_DealerDesk_Assignment (1)\VIKMO DealerDesk Assignment\ml\forecast.csv with 160 rows!


,sku,week_start,forecast_units
0,BRK-1019,2026-09-28,68.4
1,BRK-1021,2026-09-28,12.2
2,BRK-1036,2026-09-28,28.9
3,BRK-1060,2026-09-28,14.6
4,BRK-1085,2026-09-28,72.0
5,BRK-1088,2026-09-28,0.9
6,BRK-1090,2026-09-28,41.6
7,BRK-1095,2026-09-28,36.4
8,BRK-1105,2026-09-28,32.7
9,BRK-1112,2026-09-28,53.2
